In [1]:
import numpy as np
import time
import tensorflow as tf  # or tflite_runtime.interpreter
import matplotlib.pyplot as plt
from collections import namedtuple
import os
# conda activate tfTest

/Users/deqinguser/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# load tflite g_model_45.bin
interpreter = tf.lite.Interpreter(model_path="g_model_45.bin")
interpreter.allocate_tensors()
# show me input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print("Input details:", input_details)
print("Output details:", output_details)

#ifdef MODEL_45
#// input/output quantization parameters. These are extracted from the .tflite model file with print_quant_params.py
MODEL_INPUT_QUANT_SCALE = 0.10532574355602264
MODEL_INPUT_QUANT_ZERO = -74
MODEL_OUTPUT_QUANT_SCALE = 0.03067750856280327
MODEL_OUTPUT_QUANT_ZERO = 57

MODEL_THRESHOLD             = 0.385   # optimized by F1-score curve
MODEL_THRESHOLD_FOR_BG_SUB = 0.3
MODEL_NMS_THRESHOLD        = 0.3

MODEL_EMA_DECAY = 0.99

MODEL_COORD_SCALE = 4

#deqing Test

BACKGROUND_UPDATE_STEPS = 2
MODEL_EMA_DECAY = 0.8
MODEL_THRESHOLD = 0.4
MODEL_THRESHOLD_FOR_BG_SUB = 0.4

k_anchors = np.array([
    [8.853734918993446, 12.044021027232077],
    [7.7503849920115115, 6.525571303406621],
    [4.530664042752944, 7.975254206687415],
    [4.25046050720126, 5.26986904910139],
    [2.5676662786561533, 3.5365033789243747]
], dtype=np.float32)

Detection = namedtuple('Detection', ['x', 'y', 'w', 'h', 'conf'])

Input details: [{'name': 'input_1_int8', 'index': 0, 'shape': array([ 1, 24, 32,  1], dtype=int32), 'shape_signature': array([-1, 24, 32,  1], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.10532574355602264, -74), 'quantization_parameters': {'scales': array([0.10532574], dtype=float32), 'zero_points': array([-74], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
Output details: [{'name': 'Identity_int8', 'index': 51, 'shape': array([ 1,  6,  8, 25], dtype=int32), 'shape_signature': array([-1,  6,  8, 25], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.03067750856280327, 57), 'quantization_parameters': {'scales': array([0.03067751], dtype=float32), 'zero_points': array([57], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]


/Users/deqinguser/Library/Python/3.9/lib/python/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [3]:
#input data is 16bit, 100x of the celsius value, scale by *scale_factor-zero, then clamp 127~-128

loadFileName = "Serial Debug 2025-05-06 164458.txt"
allFrames = []
with open(loadFileName, "r") as f:
    lines = f.readlines()
    valuesForFrame = []
    for line in lines:
        if line.startswith("=========="):
            valuesForFrame = []
        else:
            splited = line.strip().strip(",").split(",")
            if len(splited) == 32:
                for i in range(32):
                    valuesForFrame.append(float(splited[i]))
                if len(valuesForFrame) == 32*24:
                    allFrames.append(valuesForFrame)

#drop the first 2 frames, they may not be valid
allFrames = allFrames[2:]

print("allFrames:", len(allFrames))


allFrames: 295


In [4]:
background = np.zeros((24, 32), dtype=np.int16)

In [5]:
def convertFloatTemperatureToInt16(temperatureList):
    if len(temperatureList) != 32 * 24:
        raise ValueError("Temperature list must have 768 elements (32x24).")
    int16List = []
    for temp in temperatureList:
        temp = int(temp * 100)  # scale by 100
        int16List.append(temp)
    #convert to numpy array
    int16List = np.array(int16List, dtype=np.int16).reshape((24, 32))
    return int16List

def initBackground(inputData):
    # initialize background
    # not using MODEL_BACKGROUND_INIT_STEPS (using average of start frames 3)
    # check if the shape of inputData is same
    if background.shape != inputData.shape:
        print("background shape:", background.shape)    
        print("inputData shape:", inputData.shape) 
        raise ValueError("Input data shape does not match background shape.")
    for i in range(24):
        for j in range(32):
            background[i][j] = inputData[i][j]

def getDifferenceFromGroundAsInt8(inputInt16):
    diff = inputInt16 - background
    diffScaled = diff * MODEL_INPUT_QUANT_SCALE + MODEL_INPUT_QUANT_ZERO
    clamped = np.clip(diffScaled, -128, 127).astype(np.int8)
    return clamped

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def dequant(x, q_scale, q_zero):
    # Dequantize the value using the scale and zero point
    return (x - q_zero) * q_scale


def post_get_bboxes(output_data, thresh, coord_scale):
    # Post-process the output data to get bounding boxes
    num_channels = output_data.shape[3]
    if num_channels != 5*5:
        raise ValueError("Output data must have 25 channels.")
    q_scale = MODEL_OUTPUT_QUANT_SCALE
    q_zero = MODEL_OUTPUT_QUANT_ZERO
    # conf is calculated from int8 tensor intput as: conf =  sigmoid((in_int8 - q_zero) * q_scale)
    # instead of evaluating this expression for every element in the tensor, we reverse calculate
    # what the int8 threshold should be and use that for comparison which is much faster since we
    # only need to do this once
    int_thresh = round(-np.log(1.0 / thresh - 1.0) / q_scale + q_zero);
    int8_thresh = np.clip(int_thresh, -128, 127).astype(np.int8)
    #print("int8_thresh:", int8_thresh)

    detections = []

    for y in range(6):
        for x in range(8):
            for anchor in range(5):
                int8_objectness = output_data[0][y][x][anchor * 5 + 4]
                if int8_objectness < int8_thresh:
                    continue
                oneDetectionConf = sigmoid(dequant(int8_objectness, q_scale, q_zero))

                #dequant box coordinates
                pred_x = dequant(output_data[0][y][x][anchor * 5 + 0], q_scale, q_zero)
                pred_y = dequant(output_data[0][y][x][anchor * 5 + 1], q_scale, q_zero)
                pred_w = dequant(output_data[0][y][x][anchor * 5 + 2], q_scale, q_zero)
                pred_h = dequant(output_data[0][y][x][anchor * 5 + 3], q_scale, q_zero)
                #extract box coordinates
                bbox_x = (x + sigmoid(pred_x)) * coord_scale
                bbox_y = (y + sigmoid(pred_y)) * coord_scale
                bbox_w = (k_anchors[anchor][0] * np.exp(pred_w)) * coord_scale
                bbox_h = (k_anchors[anchor][1] * np.exp(pred_h)) * coord_scale
                #convert center xy to top left xy
                oneDetectionX = bbox_x - bbox_w / 2.0
                oneDetectionY = bbox_y - bbox_h / 2.0
                oneDetectionW = bbox_w
                oneDetectionH = bbox_h
                oneDetection = Detection(oneDetectionX, oneDetectionY, oneDetectionW, oneDetectionH, oneDetectionConf)

                detections.append(oneDetection)

    return detections

def post_box_iou(a, b):
    # calculate box areas
    area_a = a.w * a.h
    area_b = b.w * b.h
    if area_a <= 0 or area_b <= 0:
        return 0.0

    # calculate intersection area
    intersection_ymin = max(a.y, b.y)
    intersection_xmin = max(a.x, b.x)
    intersection_ymax = min(a.y + a.h, b.y + b.h)
    intersection_xmax = min(a.x + a.w, b.x + b.w)
    intersection_area = max(intersection_ymax - intersection_ymin, 0) * max(intersection_xmax - intersection_xmin, 0)

    return intersection_area / (area_a + area_b - intersection_area)

def post_nms(detections, thresh):
    # Non-Maximum Suppression (NMS) to filter out overlapping boxes
    # sort detections by confidence in descending order
    detections.sort(key=lambda x: x.conf, reverse=True)

    suppressed = [False] * len(detections)
    for i in range(len(detections)):
        if suppressed[i]:
            continue
        for j in range(i + 1, len(detections)):
            if suppressed[j]:
                continue
            if post_box_iou(detections[i], detections[j]) > thresh:
                suppressed[j] = True

    return [d for i, d in enumerate(detections) if not suppressed[i]]

def background_update(inputInt16,detections,ema_decay,detectionThreshold):
    mask = np.zeros((24, 32), dtype=np.int16)
    maskMargin = 1
    #iterate over all detections, change Mask to 1 when detection is valid
    for detection in detections:
        #check if the detection is valid
        if (detection.conf > detectionThreshold) and (detection.w > 0) and (detection.h > 0):
            #calculate the box coordinates
            x1 = int(detection.x - maskMargin)
            y1 = int(detection.y - maskMargin)
            x2 = int(detection.x + detection.w + maskMargin)
            y2 = int(detection.y + detection.h + maskMargin)
            #clamp the coordinates to the image size
            x1 = max(0, x1)
            y1 = max(0, y1)
            x2 = min(32, x2)
            y2 = min(24, y2)
            #set the mask to 1
            mask[y1:y2, x1:x2] = 1
    
    for y in range(24):
        for x in range(32):
            if mask[y][x] == 1:
                #inputValue = background[y][x]
                pass    
            else:
                inputValue = inputInt16[y][x]
                #update the background with EMA
                background[y][x] = int(ema_decay * background[y][x] + (1 - ema_decay) * inputValue)
            
    


In [ ]:
convertedInt16Frames = [convertFloatTemperatureToInt16(frame) for frame in allFrames]
initBackground(convertedInt16Frames[0])

backgroundCounter = 0

def feedInput(FrameData):
    diffInt8 = getDifferenceFromGroundAsInt8(FrameData)
    # set input tensor
    input_data = diffInt8.reshape((1, 24, 32, 1)).astype(np.int8)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    # run inference
    interpreter.invoke()
    # get output tensor
    output_data = interpreter.get_tensor(output_details[0]['index'])
    #print("output_data shape:", output_data.shape)
    modelThreshold = min(MODEL_THRESHOLD_FOR_BG_SUB,MODEL_THRESHOLD)
    detections = post_get_bboxes(output_data, modelThreshold, MODEL_COORD_SCALE)
    #print("len(detections):", len(detections))
    #print("detections:", detections)
    detections = post_nms(detections, MODEL_NMS_THRESHOLD)
    #print("len(detections) post_nms:", len(detections))
    #print("detections:", detections)
    #convert convertedInt16Frames to an image and save it

    global backgroundCounter
    if backgroundCounter >= BACKGROUND_UPDATE_STEPS:
        #update background every n frames
        background_update(FrameData, detections, MODEL_EMA_DECAY, MODEL_THRESHOLD_FOR_BG_SUB)
        backgroundCounter = 0
        # print("background updated")
        # print("background", background[0][0])
            

    backgroundCounter += 1


    thermal_data = convertedInt16Frames[i]
    min_temp = 25*100
    max_temp = 35*100
    thermal_data = np.clip(thermal_data, min_temp, max_temp)  # Clip the values to the range
    thermal_data = (thermal_data - min_temp) * 255.0/ (max_temp - min_temp)   # Normalize to 0-255
    thermal_data = thermal_data.astype(np.uint8)  # Convert to uint8

    colormap = plt.get_cmap('jet', 256)  # Use 'jet' colormap with 256 colors
    color_image = colormap(thermal_data)[:, :, :3]  # Get RGB values (ignore alpha channel)
    color_image = (color_image * 255).astype(np.uint8)  # Convert to uint8
    scaleFactor = 32
    #scale up to 16x
    color_image = np.repeat(np.repeat(color_image, scaleFactor, axis=0), scaleFactor, axis=1)
    #draw the boxes on the image
    for detection in detections:
        print("detection:", detection)
        x = int(detection.x * scaleFactor)
        y = int(detection.y * scaleFactor)
        w = int(detection.w * scaleFactor)
        h = int(detection.h * scaleFactor)
        conf = detection.conf
        #draw the box on the image, on sides only

        x = np.clip(x, 0, color_image.shape[1]-1)
        y = np.clip(y, 0, color_image.shape[0]-1)
        w = np.clip(w, 0, color_image.shape[1]-x-1)
        h = np.clip(h, 0, color_image.shape[0]-y-1)

        conf = np.clip(conf, 0, 1)
        #map the confidence to a color, 0.3 is red, 0.8 or more is green
        confMap = np.clip((conf - 0.3) / (0.8 - 0.3), 0, 1)
        confMap = confMap * 0.5 + 0.5
        color = (1-confMap) * np.array([1, 0, 0]) + confMap * np.array([0, 1, 0])
        color = (color * 255).astype(np.uint8)
        # print("color:", color)
        color_image[y:y+h, x:x+1, :] = color
        color_image[y:y+h, x+w-1:x+w, :] = color
        color_image[y:y+1, x:x+w, :] = color
        color_image[y+h-1:y+h, x:x+w, :] = color

    #save the image
    plt.imsave(f"output/output_{i:03d}.jpg", color_image)
    #  ffmpeg -framerate 10 -i output/output_%03d.jpg -c:v libx264 -pix_fmt yuv420p output.mp4

for i in range(len(convertedInt16Frames)):
    if (i<30):
        diffInt8ToSave = getDifferenceFromGroundAsInt8(convertedInt16Frames[i])
        #print(diffInt8ToSave)
        # save the diffInt8ToSave to a file
        with open(f"output/diffInt8_{i:03d}.bin", "wb") as f:
            f.write(diffInt8ToSave.tobytes())
    feedInput(convertedInt16Frames[i])

    print(i)
    # if i > 3:
    #     break

#run ffmpeg -framerate 10 -i output/output_%03d.jpg -c:v libx264 -pix_fmt yuv420p output.mp4
#os.system("ffmpeg -framerate 10 -i output/output_%03d.jpg -c:v libx264 -pix_fmt yuv420p output.mp4")


[[-74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74
  -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74]
 [-74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74
  -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74]
 [-74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74
  -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74]
 [-74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74
  -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74]
 [-74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74
  -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74]
 [-74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74
  -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74]
 [-74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74
  -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74]
 [-74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74 -74